# Fine-tuning Whisper for Telugu ASR

Fine-tunes `openai/whisper-small` on Telugu speech from Google FLEURS and reports
Word Error Rate (WER) and Character Error Rate (CER) **before vs. after** fine-tuning.

**Run on Colab:** Runtime -> Change runtime type -> **T4 GPU**, then Run all (about 45-60 min).

In [ ]:
!pip -q install "datasets==3.6.0" transformers accelerate jiwer librosa soundfile matplotlib

In [ ]:
import json, random, torch
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from datasets import load_dataset, Audio
from transformers import (WhisperProcessor, WhisperForConditionalGeneration,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments)
import jiwer

assert torch.cuda.is_available(), "Enable a GPU: Runtime -> Change runtime type -> T4 GPU"
device = "cuda"

MODEL_ID  = "openai/whisper-small"
LANGUAGE  = "telugu"
FLEURS_CFG = "te_in"
N_TRAIN, N_TEST = 1500, 200
MAX_STEPS = 600
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## 1. Data (Google FLEURS, Telugu)

In [ ]:
train_ds = load_dataset("google/fleurs", FLEURS_CFG, split="train", trust_remote_code=True)
test_ds  = load_dataset("google/fleurs", FLEURS_CFG, split="test",  trust_remote_code=True)

train_ds = train_ds.shuffle(seed=SEED).select(range(min(N_TRAIN, len(train_ds))))
test_ds  = test_ds.shuffle(seed=SEED).select(range(min(N_TEST, len(test_ds))))

keep = ["audio", "transcription"]
train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c not in keep])
test_ds  = test_ds.remove_columns([c for c in test_ds.column_names if c not in keep])

train_ds = train_ds.cast_column("audio", Audio(sampling_rate=16000))
test_ds  = test_ds.cast_column("audio", Audio(sampling_rate=16000))
print(train_ds, test_ds, sep="\n")
print("Example:", train_ds[0]["transcription"])

## 2. Model and baseline (zero-shot) evaluation

In [ ]:
processor = WhisperProcessor.from_pretrained(MODEL_ID, language=LANGUAGE, task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID).to(device)
model.generation_config.language = LANGUAGE
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None
model.config.forced_decoder_ids = None

def transcribe(model, dataset, batch_size=16):
    model.eval()
    preds = []
    for i in range(0, len(dataset), batch_size):
        batch = dataset[i:i + batch_size]
        arrays = [a["array"] for a in batch["audio"]]
        feats = processor.feature_extractor(arrays, sampling_rate=16000,
                                            return_tensors="pt").input_features.to(device)
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            ids = model.generate(feats, language=LANGUAGE, task="transcribe", max_new_tokens=225)
        preds += processor.batch_decode(ids, skip_special_tokens=True)
    return [p.strip() for p in preds]

def score(refs, preds):
    return {"WER": round(100 * jiwer.wer(refs, preds), 2),
            "CER": round(100 * jiwer.cer(refs, preds), 2)}

refs = [t.strip() for t in test_ds["transcription"]]
base_preds = transcribe(model, test_ds)
base_scores = score(refs, base_preds)
print("Baseline (zero-shot Whisper-small):", base_scores)

## 3. Prepare training data

In [ ]:
def prepare(example):
    audio = example["audio"]
    example["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
    example["labels"] = processor.tokenizer(example["transcription"]).input_ids
    return example

train_proc = train_ds.map(prepare, remove_columns=train_ds.column_names)

@dataclass
class DataCollator:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]):
        inputs = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(inputs, return_tensors="pt")
        label_feats = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_feats, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

collator = DataCollator(processor, model.config.decoder_start_token_id)

## 4. Fine-tune

In [ ]:
model.config.use_cache = False
args = Seq2SeqTrainingArguments(
    output_dir="whisper-small-te",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=50,
    max_steps=MAX_STEPS,
    gradient_checkpointing=True,
    fp16=True,
    logging_steps=25,
    eval_strategy="no",
    save_strategy="no",
    report_to="none",
    remove_unused_columns=False,
    label_names=["labels"],
    seed=SEED,
)
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_proc, data_collator=collator)
trainer.train()

## 5. Evaluate after fine-tuning

In [ ]:
model.config.use_cache = True
ft_preds = transcribe(model, test_ds)
ft_scores = score(refs, ft_preds)
print("Baseline  :", base_scores)
print("Fine-tuned:", ft_scores)

results = {"model": MODEL_ID, "dataset": f"google/fleurs {FLEURS_CFG}",
           "n_train": len(train_ds), "n_test": len(test_ds), "max_steps": MAX_STEPS,
           "baseline": base_scores, "finetuned": ft_scores}
json.dump(results, open("results.json", "w"), indent=2)

fig, ax = plt.subplots(figsize=(5, 3.5))
x = np.arange(2); w = 0.35
ax.bar(x - w/2, [base_scores["WER"], base_scores["CER"]], w, label="Zero-shot")
ax.bar(x + w/2, [ft_scores["WER"], ft_scores["CER"]], w, label="Fine-tuned")
ax.set_xticks(x); ax.set_xticklabels(["WER", "CER"]); ax.set_ylabel("Error rate (%)")
ax.set_title("Whisper-small on Telugu (FLEURS test)"); ax.legend()
plt.tight_layout(); plt.savefig("wer_comparison.png", dpi=150); plt.show()

In [ ]:
for i in range(5):
    print(f"--- sample {i} ---")
    print("REF :", refs[i])
    print("BASE:", base_preds[i])
    print("FT  :", ft_preds[i])

## 6. Save (optional)
Run the cell below to download `results.json` and `wer_comparison.png`, then add them to your GitHub repo.

In [ ]:
from google.colab import files
files.download("results.json"); files.download("wer_comparison.png")